## Purpose:
Train and evaluate an XGBoost classifier to predict in-hospital mortality for ICU patients using SQL-engineered features from the MIMIC-III demo dataset. This notebook serves as the  proof-of-concept for Track 1 from raw database query to a trained, explainable ML model.

## Context:
The MIMIC-III demo contains ~100 ICU stays with a high mortality rate relative to the full dataset, creating a significant class imbalance. Standard accuracy metrics are misleading under these conditions, so this notebook uses AUROC as the primary evaluation metric and applies scale_pos_weight in XGBoost to compensate for the skewed class distribution.
All features are pulled directly from the SQL views built in sql/features.sql via tracks/sql_features.py — no feature engineering happens inside this notebook. The notebook's job is purely modeling: load features, handle imbalance, train, evaluate, and explain.
This notebook is intentionally scoped to the demo dataset as a pipeline prototype. Results should be interpreted as proof-of-concept only — production-quality performance requires the full MIMIC-III dataset (~46,000 ICU stays).

# Section 1 Imports

In [ ]:
# Data handling
import pandas as pd
import numpy as np 

# Database
import sqlite3

# Model 
from xgboost import XGBClassifier

# Class imbalance 
from imblearn.over_sampling import SMOTE

# Sklearn utilities 
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.preprocessing import StandardScaler

# Explainability 
import shap

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Extras 
import warnings
from pathlib import Path

# Section 2 Configuration 

In [ ]:
# Setup path to mimic.db 
mimic = Path("../data/mimic.db")

# Setup Random seeds 
seed = 42 
np.random_seed(seed)